# Aproksimacioni algoritmi za Steinerova stabla u težinskim grafovima

**Nikola Labus**

Naučno izračunavanje 2025/26 — Matematički fakultet, Univerzitet u Beogradu

---

Implementacija, evaluacija i poređenje 7 algoritama na SteinLib benchmark instancama

## Problem Steinerovog stabla

**Ulaz:** Težinski graf G = (V, E, w) i skup terminala T ⊆ V

**Cilj:** Pronaći stablo minimalne težine koje povezuje sve terminale, koristeći po potrebi i neterminalne čvorove (Steinerove tačke)

### Zašto je ovo teško?

- Problem je **NP-težak** — ne postoji poznat polinomijalni algoritam za egzaktno rešenje
- Brute force: O(2^(n-t)) — eksponencijalan u broju neterminalnih čvorova
- Na 30 čvorova sa 8 terminala: 2²² ≈ 4 miliona podskupova → **~13 minuta**
- Na 100 čvorova: praktično nemoguće

### Primene

- Dizajn VLSI čipova (povezivanje pinova)
- Telekomunikacione mreže (minimalna infrastruktura)
- Bioinformatika (protein-protein interakcione mreže)

## Implementirani algoritmi

| # | Algoritam | Tip | Faktor | Složenost | Godina |
|---|---|---|---|---|---|
| 1 | **Brute Force** | Egzaktan | Optimum | O(2^(n-t)) | — |
| 2 | **MST Heuristika** (Kou–Markowsky–Berman) | Aproksimacija | 2 | O(t²·n log n) | 1981 |
| 3 | **Mehlhorn** (Voronoi) | Aproksimacija | 2 | O(m log n) | 1988 |
| 4 | **SPH** (Takahashi–Matsuyama) | Aproksimacija | 2 | O(t·V_tree·n log n) | 1980 |
| 5 | **Zelikovsky** | Aproksimacija | 11/6 ≈ 1.83 | O(t³·n) | 1993 |
| 6 | **Zel + SPH** | Hibrid | 11/6 | O(t³·n) + SPH | — |
| 7 | **Zel + Mehlhorn** | Hibrid | 11/6 | O(t³·n) + Mehlhorn | — |

Svi algoritmi implementirani u **Python-u** koristeći **NetworkX** biblioteku.

## Ključne ideje algoritama (1/2)

### MST Heuristika
1. Za svaki par terminala izračuna najkraći put (Dijkstra)
2. Gradi kompletni graf terminala sa tim rastojanjima
3. MST kompletnog grafa → zameni grane putevima → MST ponovo → pruning

### Mehlhorn — ubrzanje preko Voronoi particije
1. Multi-source Dijkstra iz svih terminala — svaki čvor dobija labelu najbližeg terminala
2. Za grane između različitih Voronoi regiona računa cenu povezivanja
3. MST međuregionalnog grafa → rekonstrukcija puteva → pruning
4. **Efekat:** umesto t² Dijkstra poziva, jedan prolaz → 1000–5000× brži

### SPH — greedy pristup
1. Počinje sa jednim terminalom
2. U svakom koraku dodaje najbliži nepokriven terminal
3. Ponavlja dok svi terminali nisu u stablu

## Ključne ideje algoritama (2/2)

### Zelikovsky — jedini sa faktorom boljim od 2

1. Kreće od početnog rešenja (MST heuristika, SPH, ili Mehlhorn)
2. Ispituje sve **trojke terminala** (t₁, t₂, t₃)
3. Za svaki neterminalni čvor v računa cenu **zvezde**: dist(v, t₁) + dist(v, t₂) + dist(v, t₃)
4. Ako je zvezda jeftinija od trenutnog povezivanja — primenjuje poboljšanje
5. Ponavlja dok ima ušteda

**Problem:** O(t³·n) po iteraciji — na 50+ terminala i 500 čvorova postaje veoma spor

### Hibridni pristupi

- **Zel+SPH:** SPH daje bolji početni kvalitet → manje iteracija, ali i manje prostora za poboljšanje
- **Zel+Mehlhorn:** Mehlhorn daje brže početno rešenje → ukupno brži, ali drugačija topologija

## Skupovi podataka — SteinLib benchmark

| Dataset | Instanci | Čvorovi | Grane | Terminali | Svrha |
|---|---|---|---|---|---|
| **Small** | 10 | 5–30 | 7–90 | 3–8 | Verifikacija sa brute force |
| **B** (Beasley) | 18 | 50–100 | 63–200 | 9–50 | Srednji grafovi |
| **C** (Beasley) | 20 | 500 | 625–12,500 | 5–250 | Veliki grafovi |
| **C_30** | 20 | 500 | — | max 30 | Skalabilnost Zelikovskog |
| **C_50** | 20 | 500 | — | max 50 | Granica primenljivosti |

- Format: **STP** (Steiner Tree Problem) — standardni format iz SteinLib kolekcije
- C_30 i C_50 su redukovane verzije C dataseta — isti grafovi, manji broj terminala
- Redukcija omogućava testiranje Zelikovsky algoritma koji eksplodira na 100+ terminala

## Demo — pokretanje na primeru

Instanca `small01`: 5 čvorova, 7 grana, 3 terminala (1, 3, 5)

In [ ]:
%run 01_algorithms.ipynb

In [ ]:
G, terminals, name = parse_stp('data/small/small01.stp')
print(f'Instanca: {name}')
print(f'Čvorovi: {G.number_of_nodes()}, Grane: {G.number_of_edges()}')
print(f'Terminali: {terminals}')
print(f'Grane: {[(u, v, d["weight"]) for u, v, d in G.edges(data=True)]}')

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

pos = nx.spring_layout(G, seed=42)
terminal_set = set(terminals)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# 1 - Originalni graf
ax = axes[0]
ax.set_title('Originalni graf', fontsize=13, fontweight='bold')
colors = ['#e74c3c' if n in terminal_set else '#3498db' for n in G.nodes()]
nx.draw(G, pos, ax=ax, with_labels=True, node_color=colors, node_size=500,
        font_color='white', font_weight='bold', edge_color='#888', width=1.5)
labels = nx.get_edge_attributes(G, 'weight')
nx.draw_networkx_edge_labels(G, pos, edge_labels=labels, ax=ax, font_size=10)
red_patch = mpatches.Patch(color='#e74c3c', label='Terminal')
blue_patch = mpatches.Patch(color='#3498db', label='Neterminal')
ax.legend(handles=[red_patch, blue_patch], loc='lower left', fontsize=9)

# 2 - Brute Force (optimalno)
bf_tree, bf_weight, _ = brute_force_steiner(G, terminals)
ax = axes[1]
ax.set_title(f'Brute Force (OPT) — težina: {bf_weight}', fontsize=13, fontweight='bold')
nx.draw(G, pos, ax=ax, with_labels=True, node_color='#ddd', node_size=500,
        font_color='#999', edge_color='#ddd', width=1, style='dashed')
bf_colors = ['#e74c3c' if n in terminal_set else '#2ecc71' for n in bf_tree.nodes()]
nx.draw(bf_tree, pos, ax=ax, with_labels=True, node_color=bf_colors, node_size=500,
        font_color='white', font_weight='bold', edge_color='#2ecc71', width=3)
bf_labels = nx.get_edge_attributes(bf_tree, 'weight')
nx.draw_networkx_edge_labels(bf_tree, pos, edge_labels=bf_labels, ax=ax, font_size=10)

# 3 - MST Heuristika
mst_tree, mst_weight = mst_heuristic(G, terminals)
ax = axes[2]
ax.set_title(f'MST Heuristika — težina: {mst_weight}', fontsize=13, fontweight='bold')
nx.draw(G, pos, ax=ax, with_labels=True, node_color='#ddd', node_size=500,
        font_color='#999', edge_color='#ddd', width=1, style='dashed')
mst_colors = ['#e74c3c' if n in terminal_set else '#9b59b6' for n in mst_tree.nodes()]
nx.draw(mst_tree, pos, ax=ax, with_labels=True, node_color=mst_colors, node_size=500,
        font_color='white', font_weight='bold', edge_color='#9b59b6', width=3)
mst_labels = nx.get_edge_attributes(mst_tree, 'weight')
nx.draw_networkx_edge_labels(mst_tree, pos, edge_labels=mst_labels, ax=ax, font_size=10)

plt.tight_layout()
plt.show()

In [ ]:
import time

algorithms = [
    ('Brute Force', lambda G, t: brute_force_steiner(G, t)[:2]),
    ('MST Heuristika', mst_heuristic),
    ('Mehlhorn', mehlhorn_steiner),
    ('SPH', sph_steiner),
    ('Zelikovsky', zelikovsky_steiner),
    ('Zel+SPH', zelikovsky_sph),
    ('Zel+Mehlhorn', zelikovsky_mehlhorn),
]

print(f'{"Algoritam":<20} {"Težina":>8} {"Odnos":>8} {"Vreme":>10}')
print('-' * 48)

opt_weight = None
for name, algo in algorithms:
    start = time.time()
    _, w = algo(G, terminals)
    elapsed = time.time() - start
    if opt_weight is None:
        opt_weight = w
    ratio = w / opt_weight
    print(f'{name:<20} {w:>8.0f} {ratio:>8.4f} {elapsed:>10.4f}s')

## Rezultati — kvalitet rešenja

Prosečan odnos težine prema referentnom algoritmu (niže = bolje):

| Algoritam | Small (vs OPT) | B (vs MST) | C-30 (vs MST) | Rang |
|---|---|---|---|---|
| **SPH** | **1.0226** | **0.9871** | **0.9589** | **1.** |
| Zel+SPH | 1.0226 | 0.9840 | 0.9579 | 2. |
| Zel+Mehlhorn | 1.0301 | 0.9925 | 0.9997 | 3. |
| Zelikovsky | 1.0385 | 0.9922 | 0.9935 | 4. |
| Mehlhorn | 1.0588 | 0.9984 | 1.0063 | 5. |
| MST Heuristika | 1.0840 | 1.0000 | 1.0000 | 6. |

SPH i Zel+SPH konzistentno daju **2–4% bolje rezultate** od MST heuristike.

## Rezultati — brzina izvršavanja

| Algoritam | Small | B (100 čv) | C (500 čv, 30 t) | Skalabilnost |
|---|---|---|---|---|
| **Mehlhorn** | ~0.0006s | ~0.003s | ~0.01s | Izuzetna |
| MST Heuristika | ~0.001s | ~0.7s | ~28s | Dobra |
| Zelikovsky | ~0.1s | ~3.7s | ~104s | Loša |
| SPH | ~0.005s | ~8.6s | ~169s | Loša |
| Zel+SPH | ~0.09s | ~9.3s | ~243s | Najlošija |
| Brute Force | ~763s | — | — | Neprimenljiv |

Mehlhorn je **1000–5000× brži** od ostalih na velikim grafovima zahvaljujući Voronoi particiji.

## Ključni uvidi

### 1. Teorija ≠ Praksa
Zelikovsky (faktor 11/6 ≈ 1.83) bi trebalo da bude bolji od SPH (faktor 2), ali u praksi **SPH često daje bolje rezultate** jer greedy pristup dobro eksploatiše strukturu realnih grafova.

### 2. Mehlhorn dominira po brzini
Voronoi particija svodi t² Dijkstra poziva na jedan prolaz — **O(m log n)** umesto **O(t²·n log n)**. Na C datasetu ostaje ispod 1 sekunde dok drugi algoritmi prelaze minute.

### 3. Izbor baznog algoritma za Zelikovsky je bitan
- Zel+SPH → najbolji kvalitet, ali najsporiji
- Zel+Mehlhorn → brži, ali nestabilan na većim grafovima
- Zel+MST → najstabilniji kompromis

### 4. Skalabilnost je glavni ograničavajući faktor
Sa porastom terminala, razlike u kvalitetu rastu, ali razlike u vremenu **eksplodiraju** — na 50 terminala i 500 čvorova, SPH traje 100–200× duže od Mehlhorn-a.

## Preporuke po scenariju

| Scenarijo | Preporučen algoritam | Zašto |
|---|---|---|
| Mali grafovi (≤30 čv) | **Brute Force** | Egzaktan, završava za sekunde |
| Srednji (50–100 čv) | **Zelikovsky** | Najbolji kompromis kvalitet/vreme |
| Veliki, brzina prioritet | **Mehlhorn** | Jedini ispod 1s, kvalitet ±5% |
| Veliki, kvalitet prioritet | **SPH** ili **Zel+SPH** | Do 4% bolje, ali minutima po instanci |
| Praktičan kompromis | **Zelikovsky + MST** | Stabilan, bez dramatičnog usporavanja |

## Izazovi na koje se naišlo

### Skalabilnost Zelikovskog
- Na C datasetu (500 čvorova, 100 terminala) Zelikovsky bi trajao satima
- **Rešenje:** Kreiranje redukovanih verzija dataseta (C_30, C_50) sa ograničenim brojem terminala

### Paralelizacija na Windows-u
- `ProcessPoolExecutor` ne radi u Jupyter-u na Windows-u zbog `fork()` ograničenja
- **Rešenje:** `ThreadPoolExecutor` na Windows-u, `ProcessPoolExecutor` na Linux-u

### Odabir referentnog rezultata
- Za Small set: poređenje sa brute force optimumom
- Za B i C set: brute force neprimenljiv → poređenje sa MST heuristikom kao bazom
- Zelikovsky varijante ponekad daju **bolje od baze** (odnos < 1.0)

## Tehnologije

- **Python 3** — implementacija svih algoritama
- **NetworkX** — grafovske strukture, Dijkstra, MST, shortest path
- **Matplotlib** — vizuelizacija rezultata (histogrami, scatter plotovi, bar chart-ovi)
- **Jupyter Notebook** — interaktivna dokumentacija i reprodukovanje rezultata
- **SteinLib** — standardizovani benchmark skupovi podataka (STP format)
- **Git / GitHub** — verzionisanje koda

## Literatura

1. **Dinitz, M.** (2026). *Lecture 2: Steiner Tree, TSP.* 601.435/635 Approximation Algorithms, Johns Hopkins University.

2. **Kou, L., Markowsky, G., Berman, L.** (1981). *A fast algorithm for Steiner trees.* Acta Informatica, 15(2), 141–145.

3. **Mehlhorn, K.** (1988). *A faster approximation algorithm for the Steiner problem in graphs.* Information Processing Letters, 27(3), 125–128.

4. **Takahashi, H., Matsuyama, A.** (1980). *An approximate solution for the Steiner problem in graphs.* Math. Japonica, 24(6), 573–577.

5. **Zelikovsky, A. Z.** (1993). *An 11/6-approximation algorithm for the network Steiner problem.* Algorithmica, 9(5), 463–470.

6. **Pajor, T., Uchoa, E., Werneck, R. F.** (2018). *Strong Steiner Tree Approximations in Practice.* arXiv:1409.8318.

7. **Huang, S. Y. et al.** (2013). *Steiner tree methods for optimal sub-network identification.* BMC Bioinformatics, 14, 144.

## Hvala na pažnji!

**Pitanja?**

---

Kod i rezultati dostupni na GitHub-u:

`github.com/nikolalabus/Approxiation-algorithms-for-Steiner-trees-in-weighted-graphs`